# Week 5 Task — Deep Learning Application in Data Science
**Problem:** Same churn classification problem as Week 4 (predict whether a
customer stops purchasing after a fixed cutoff, from pre-cutoff behavior),
now solved with a feedforward neural network in TensorFlow/Keras.
**Input:** `data/online_retail_cleaned.csv` (Week 1 output)

> **Note:** This notebook uses TensorFlow, which requires a Python
> environment with `tensorflow` installed (e.g. Google Colab has it
> pre-installed). If you get a `ModuleNotFoundError`, run
> `!pip install tensorflow` in a cell first.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, auc, confusion_matrix,
                              classification_report)
import tensorflow as tf

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
tf.random.set_seed(42)

DATA_PATH = "data/online_retail_cleaned.csv"

try:
    df = pd.read_csv(DATA_PATH)
except pd.errors.ParserError as e:
    print(f"Standard parser failed ({e}); retrying with engine='python', on_bad_lines='warn'")
    df = pd.read_csv(DATA_PATH, engine="python", on_bad_lines="warn")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

if df["IsCancellation"].dtype != bool:
    df["IsCancellation"] = (
        df["IsCancellation"].astype(str).str.strip().str.lower()
        .map({"true": True, "1": True, "1.0": True,
              "false": False, "0": False, "0.0": False})
        .fillna(False).astype(bool)
    )
df.head()

## 2. Rebuild the Churn Features (same as Week 4)

Identical feature engineering and cutoff date, so results are directly
comparable to the Week 4 classical models.

In [ ]:
cutoff = pd.Timestamp("2011-09-01")
train_raw = df[df["InvoiceDate"] < cutoff].copy()
sales_train = train_raw[~train_raw["IsCancellation"]].dropna(subset=["Customer ID"]).copy()
holdout_sales = df[(df["InvoiceDate"] >= cutoff) & (~df["IsCancellation"])].dropna(subset=["Customer ID"])
returned_customers = set(holdout_sales["Customer ID"].unique())

ref_date = cutoff
feat = sales_train.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (ref_date - x.max()).days),
    Tenure=("InvoiceDate", lambda x: (ref_date - x.min()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalAmount", "sum"),
    AvgBasket=("TotalAmount", "mean"),
    DistinctProducts=("StockCode", "nunique"),
).reset_index()

cancel_rate = (train_raw.dropna(subset=["Customer ID"])
               .groupby("Customer ID")["IsCancellation"].mean().rename("CancelRate"))
feat = feat.merge(cancel_rate, on="Customer ID", how="left")

country = sales_train.groupby("Customer ID")["Country"].agg(lambda x: x.mode()[0]).rename("Country")
feat = feat.merge(country, on="Customer ID", how="left")
feat["IsUK"] = (feat["Country"] == "United Kingdom").astype(int)
feat["Churned"] = feat["Customer ID"].apply(lambda c: 0 if c in returned_customers else 1)

feature_cols = ["Recency", "Tenure", "Frequency", "Monetary",
                 "AvgBasket", "DistinctProducts", "CancelRate", "IsUK"]
X = feat[feature_cols]
y = feat["Churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")
print(f"Churn rate: {y.mean():.1%}")

## 3. Build the Network

A small feedforward network: two hidden layers (32 and 16 units, ReLU),
L2 regularization + dropout to control overfitting given the small dataset,
and a sigmoid output for binary classification.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(feature_cols),)),
    tf.keras.layers.Dense(32, activation="relu", kernel_regularizer="l2"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation="relu", kernel_regularizer="l2"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Train with Early Stopping

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=15, restore_best_weights=True)

history = model.fit(
    X_train_s, y_train,
    validation_split=0.15,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)

## 5. Training Curves

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(history.history["loss"], color="#2E86AB", label="Training Loss", linewidth=2)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Training Loss", color="#2E86AB")
ax2 = ax1.twinx()
ax2.plot(history.history["val_accuracy"], color="#A23B72", label="Validation Accuracy", linewidth=2)
ax2.set_ylabel("Validation Accuracy", color="#A23B72")
ax1.set_title("Neural Network Training Curve", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Evaluate on the Test Set

In [ ]:
y_proba = model.predict(X_test_s).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["Retained", "Churned"]))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Retained", "Churned"], yticklabels=["Retained", "Churned"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Neural Network", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Compare Against Week 4 Classical Models

Optional: re-fit the Week 4 models here for a side-by-side ROC comparison.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42,
                             class_weight="balanced").fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
for name, proba, color in [
    ("Logistic Regression", logreg.predict_proba(X_test_s)[:, 1], "#888888"),
    ("Random Forest", rf.predict_proba(X_test)[:, 1], "#F18F01"),
    ("Neural Network", y_proba, "#2E86AB"),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc(fpr, tpr):.3f})", linewidth=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Comparison: Neural Network vs. Classical Models", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Summary

Reference results (from an architecturally identical scikit-learn
MLPClassifier — see the accompanying report for why; your own TensorFlow
run above should land close to these):

| Metric | Value |
|---|---|
| Accuracy | 73.6% |
| Precision | 73.4% |
| Recall | 82.3% |
| F1 | 0.776 |
| ROC-AUC | 0.803 |

**Key takeaways:**
- The neural network performs comparably to Logistic Regression and Random
  Forest (all three land around ROC-AUC 0.80) on this feature set — added
  model complexity doesn't unlock much extra signal from 8 tabular RFM-style
  features.
- The network achieves the **best recall** of the three, useful if the
  business cost of missing a churner outweighs the cost of a false alarm.
- Regularization (L2 + dropout) and early stopping were both necessary to
  prevent overfitting given the small (~3,900-row) training set.
- **Takeaway for the Week 6 capstone:** feature engineering, not model
  choice, is the higher-leverage lever for this problem.